# 07 — Changes in concordance between RNA and ADT - in regards to perturbation_2

For the most part, RNA and ADT signal correlate close to 1:1 (see fig: 06_rna_adt_concordance.png)

Does some condition (perturbation1 or 2) cause a change in concordance?
Version1: perurbation_2 culture condition affects RNA but not ADT (or vice versa)
Version2: CRISPR-KO target affects RNA but not ADT (or vice versa) for a different gene than the CRISPR target ("self" was already roughly analyzed in nb02)

We will start with Version1 here.

In [ ]:
# =============================================================================
# nb07 — RNA / ADT concordance shifts
#
# Does immune pressure change how well surface protein tracks transcript?
# For each ADT feature, we compute the per-cell Pearson correlation between
# RNA and protein separately in each condition, then ask whether that
# relationship is condition-dependent.
#
# This is a pure observation — no CRISPR layer, no signature matrix needed.
# Control-guide cells only so the CRISPR perturbations don't confound the
# RNA/protein relationship being measured.
#
# Version 2 (perturbation-driven concordance shifts) requires the ADT
# signature matrix and is deferred to nb08.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import muon as mu

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- data -----------------------------------------------------------------
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]

# remove T-cell contamination
emb   = pd.read_parquet(P.data_interim / "03_embedding.parquet")
tcell = emb.index[emb["cluster"].astype(str) == "12"]
keep  = ~rna.obs_names.isin(tcell)
rna, adt = rna[keep].copy(), adt[keep].copy()

# ---- normalise ------------------------------------------------------------
rna_n = rna.copy()
rna_n.X = rna_n.layers["counts"].copy()
sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)

adt_n = adt.copy()
adt_n.layers["counts"] = adt_n.X.copy()
mu.prot.pp.clr(adt_n, axis=cfg["protein"]["clr_margin"])

# ---- panel metadata -------------------------------------------------------
adt_to_rna = panels["adt"]["adt_to_rna"]
isotypes   = panels["adt"]["isotype_controls"]
targets    = [f for f in adt_n.var_names if f not in isotypes]
adt_annot  = panels["adt"]["annotations"]

# control-guide cells only — CRISPR layer removed
ctrl_mask = rna_n.obs[PERT].astype(str) == CTRL
print(f"control-guide cells: {ctrl_mask.sum():,}")
print(f"  Control:    {((rna_n.obs[COND].astype(str) == 'Control')    & ctrl_mask).sum():,}")
print(f"  IFNγ:       {((rna_n.obs[COND].astype(str) == 'IFNγ')       & ctrl_mask).sum():,}")
print(f"  Co-culture: {((rna_n.obs[COND].astype(str) == 'Co-culture') & ctrl_mask).sum():,}")

assert (rna_n.obs_names == adt_n.obs_names).all(), "modalities out of order"
print("\nready")

In [ ]:
sig  = pd.read_parquet(P.data_processed / "05_signatures_lfc.parquet")
padj = pd.read_parquet(P.data_processed / "05_signatures_padj.parquet")
print(f"sig: {sig.shape}")

In [ ]:
# ---- variance per feature in control cells --------------------------------
# Used to size points in the figure: low RNA variance = unreliable Pearson r
# because you are correlating two nearly-flat vectors. Point size makes the
# reliability gradient explicit without removing any feature.
var_map = {}
for adt_feat, rna_genes in adt_to_rna.items():
    rna_genes_present = [g for g in rna_genes if g in rna_n.var_names]
    if not rna_genes_present:
        continue
    mask = ctrl_mask & (rna_n.obs[COND].astype(str) == "Control")
    rna_X = rna_n[mask, rna_genes_present].X
    var_map[adt_feat] = float(np.asarray(rna_X.mean(axis=1)).ravel().var())

var_s = pd.Series(var_map).reindex(r_wide.index).fillna(0)
sizes = 15 + 185 * (var_s - var_s.min()) / (var_s.max() - var_s.min() + 1e-9)

# ---- per-cell RNA/ADT correlation per feature x condition -----------------
rows = []
for cond in cond_order:
    mask = ctrl_mask & (rna_n.obs[COND].astype(str) == cond)
    n = mask.sum()

    for adt_feat, rna_genes in adt_to_rna.items():
        if adt_feat not in adt_n.var_names:
            continue
        rna_genes_present = [g for g in rna_genes if g in rna_n.var_names]
        if not rna_genes_present:
            continue

        rna_X = rna_n[mask, rna_genes_present].X
        rna_vals = np.asarray(rna_X.mean(axis=1)).ravel()

        adt_X = adt_n[mask, adt_feat].X
        adt_vals = np.asarray(
            adt_X.todense() if sp.issparse(adt_X) else adt_X
        ).ravel()

        r, pval = stats.pearsonr(rna_vals, adt_vals)

        rows.append({
            "adt_feature": adt_feat,
            "condition": cond,
            "n_cells": int(n),
            "pearson_r": float(r),
            "pvalue": float(pval),
            "rna_var": var_map.get(adt_feat, 0.0),
            "annotation": adt_annot.get(adt_feat, ""),
        })

corr_df = pd.DataFrame(rows)

r_wide = corr_df.pivot(index="adt_feature", columns="condition",
                       values="pearson_r").reindex(columns=cond_order)
print(r_wide.round(3).sort_values("Control", ascending=False).to_string())

In [ ]:
# =============================== FIGURE ====================================
# Per-feature RNA/ADT concordance shift under immune pressure.
# Each point is one ADT feature; x = Pearson r in Control, y = r in the
# comparison condition. Diagonal = no change. Below = concordance weakened.
#
# Point size encodes RNA variance in control cells — small points have
# low variance and therefore unreliable r estimates. Interpret them with
# caution; their position may reflect noise rather than biology.
#
# Control-guide cells only: CRISPR layer removed so only the condition
# effect on RNA/protein coupling is visible.

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)

comparisons = [("Control", "IFNγ", "A"), ("Control", "Co-culture", "B")]

for a, (ref, cond, panel_label) in zip(axes, comparisons):
    x = r_wide[ref]
    y = r_wide[cond]
    feat_sizes = sizes.reindex(r_wide.index).fillna(15)

    lim = max(abs(np.concatenate([x.dropna(), y.dropna()]))) * 1.12

    # reference lines
    a.plot([-lim, lim], [-lim, lim], ls="--", c="grey", lw=1, zorder=1,
           label="no change")
    a.axhline(0, c="k", lw=0.5, ls=":", zorder=1)
    a.axvline(0, c="k", lw=0.5, ls=":", zorder=1)

    # colour by direction of shift
    colors = ["#2a7a2a" if yi > xi else "#b8860b"
              for xi, yi in zip(x, y)]

    sc = a.scatter(x, y, s=feat_sizes.values, c=colors,
                   edgecolor="white", linewidth=0.5, zorder=3, alpha=0.9)

    # label all features
    texts = [a.text(x[f], y[f], f, fontsize=7)
             for f in r_wide.index if not np.isnan(x[f]) and not np.isnan(y[f])]
    adjust_text(texts, ax=a,
                arrowprops=dict(arrowstyle="-", lw=0.4, color="grey"),
                expand=(1.4, 1.6), force_text=(0.6, 0.8),
                ensure_inside_axes=True)

    n_up   = int((y > x).sum())
    n_down = int((y < x).sum())
    a.text(0.03, 0.97,
           f"concordance improves: {n_up}\n"
           f"concordance drops:    {n_down}",
           transform=a.transAxes, ha="left", va="top",
           fontsize=8, family="monospace",
           bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                     edgecolor="none", alpha=0.85))

    a.set_xlim(-lim, lim)
    a.set_ylim(-lim, lim)
    a.set_xlabel(f"Pearson r — {ref}", fontsize=10)
    a.set_ylabel(f"Pearson r — {cond}", fontsize=10)
    a.set_title(f"{panel_label}) Control vs {cond}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

# shared legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
fig.legend(handles=[
    Patch(facecolor="#2a7a2a", label="concordance improves"),
    Patch(facecolor="#b8860b", label="concordance drops"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#555",
           markersize=4, label="small = low RNA variance → unreliable r"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#555",
           markersize=10, label="large = high RNA variance → reliable r"),
], loc="lower center", ncol=2, fontsize=8.5, frameon=False,
   bbox_to_anchor=(0.5, -0.04))

fig.suptitle("RNA / ADT concordance shift under immune pressure\n"
             "(control-guide cells only, per-cell Pearson r)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0.07, 1, 1])
savefig(fig, "07_concordance_shift", cfg)

In [ ]:
# variance in RNA per feature per condition — low variance = unreliable r
for adt_feat, rna_genes in adt_to_rna.items():
    rna_genes_present = [g for g in rna_genes if g in rna_n.var_names]
    if not rna_genes_present:
        continue
    mask = ctrl_mask & (rna_n.obs[COND].astype(str) == "Control")
    rna_X = rna_n[mask, rna_genes_present].X
    v = float(np.asarray(rna_X.mean(axis=1)).ravel().var())
    print(f"{adt_feat:10s}  RNA var = {v:.4f}")

Now we do Version2: does CRISPR KO of a different gene, cause dis-concordance in ADT vs RNA for one of the 20 genes?

In [ ]:
# ---- ADT signature matrix -------------------------------------------------
# Per (perturbation, condition) pair, per ADT feature: Cohen's d between
# perturbed cells and condition-matched control-guide cells.
#
# Cohen's d = (mean_pert - mean_ctrl) / pooled_SD
#
# With 20 features and no pseudobulk step needed, this runs fast. The same
# condition-matched-control constraint as the RNA side: controls are never
# pooled across conditions.
#
# Minimum cells per arm is looser than the RNA guide-level minimum — 20 cells
# per (perturbation, condition) on the perturbation side. The control pool is
# large enough in every condition not to be the bottleneck.

MIN_CELLS_ADT = 20

# pull CLR matrix into a dense DataFrame once — 126k x 20, fits in memory
adt_X = adt_n[:, targets].X
adt_dense = pd.DataFrame(
    np.asarray(adt_X.todense() if sp.issparse(adt_X) else adt_X),
    index=adt_n.obs_names, columns=targets
)
adt_dense[COND] = adt_n.obs[COND].values
adt_dense[PERT] = rna_n.obs[PERT].values   # same cell order asserted above

def cohen_d(a, b):
    """Pooled-SD Cohen's d. Returns NaN if either group has zero variance."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled_sd = np.sqrt(((na - 1) * a.std() ** 2 + (nb - 1) * b.std() ** 2)
                        / (na + nb - 2))
    if pooled_sd == 0:
        return np.nan
    return (a.mean() - b.mean()) / pooled_sd

rows = []
perts = [p for p in rna_n.obs[PERT].unique() if str(p) != CTRL]

for cond in cond_order:
    cond_mask = adt_dense[COND].astype(str) == cond
    ctrl_cells = adt_dense[cond_mask & (adt_dense[PERT].astype(str) == CTRL)]

    for pert in perts:
        pert_cells = adt_dense[cond_mask & (adt_dense[PERT].astype(str) == pert)]
        if len(pert_cells) < MIN_CELLS_ADT:
            continue
        for feat in targets:
            d = cohen_d(pert_cells[feat], ctrl_cells[feat])
            rows.append({
                "perturbation": pert,
                "condition": cond,
                "adt_feature": feat,
                "cohens_d": d,
                "n_pert": len(pert_cells),
                "n_ctrl": len(ctrl_cells),
            })

adt_sig = pd.DataFrame(rows)
print(f"ADT effect table: {len(adt_sig):,} rows")
print(f"  perturbations: {adt_sig['perturbation'].nunique()}")
print(f"  features:      {adt_sig['adt_feature'].nunique()}")
print(f"  conditions:    {adt_sig['condition'].nunique()}")

# pivot to a matrix matching the RNA signature shape:
# MultiIndex (perturbation, condition) x adt_feature
adt_sig_wide = adt_sig.pivot_table(
    index=["perturbation", "condition"],
    columns="adt_feature",
    values="cohens_d"
)
print(f"\nADT signature matrix: {adt_sig_wide.shape}")
adt_sig_wide.to_parquet(P.data_processed / "07_adt_signatures.parquet")
print("saved -> 07_adt_signatures.parquet")

In [ ]:
# for the 12 targets whose gene is in both modalities, does Cohen's d go
# negative when you knock out the gene? Should mirror the RNA self-knockdown.
print("ADT self-knockdown (Cohen's d, perturbation -> its own ADT feature):\n")
for adt_feat, rna_genes in adt_to_rna.items():
    if adt_feat not in targets:
        continue
    # find which perturbation targets one of the RNA genes for this ADT
    for gene in rna_genes:
        sub = adt_sig[(adt_sig["perturbation"] == gene) &
                      (adt_sig["adt_feature"] == adt_feat)]
        if sub.empty:
            continue
        vals = sub.set_index("condition")["cohens_d"].reindex(cond_order)
        print(f"{gene:8s} -> {adt_feat:8s}  "
              + "  ".join(f"{c}: {v:+.2f}" for c, v in vals.items()))

In [ ]:
# ---- z-score both axes so 1:1 is meaningful ------------------------------
# RNA LFC: z-score across all (perturbation x condition x feature) rows
# ADT Cohen's d: same
# After this, a point on the diagonal means RNA and protein moved the same
# number of SDs from their respective means — the 1:1 prior is visible.

quad["rna_lfc_z"] = (quad["rna_lfc"] - quad["rna_lfc"].mean()) / quad["rna_lfc"].std()
quad["adt_d_z"]   = (quad["adt_d"]   - quad["adt_d"].mean())   / quad["adt_d"].std()

# also z-score the self-knockdown table for consistency when you annotate
print("z-score ranges:")
print(f"  RNA LFC z: {quad['rna_lfc_z'].min():.2f} to {quad['rna_lfc_z'].max():.2f}")
print(f"  ADT d z:   {quad['adt_d_z'].min():.2f} to {quad['adt_d_z'].max():.2f}")

In [ ]:
# re-load-in, to plot, below.

# =============================================================================
# nb07 — RNA / ADT concordance and perturbation-level RNA/protein comparison
# (reload cell)
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import muon as mu

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
CTRL       = cfg["schema"]["control_label"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- signature matrices ---------------------------------------------------
sig     = pd.read_parquet(P.data_processed / "05_signatures_lfc.parquet")
adt_sig = pd.read_parquet(P.data_processed / "07_adt_signatures.parquet")

# adt_sig was saved wide (MultiIndex rows, feature columns) — melt back to long
# so the (perturbation, condition, adt_feature, cohens_d) row form the figure
# code expects is available
if isinstance(adt_sig.index, pd.MultiIndex):
    adt_sig = (adt_sig.reset_index()
               .melt(id_vars=["perturbation", "condition"],
                     var_name="adt_feature", value_name="cohens_d"))

# ---- panel metadata -------------------------------------------------------
adt_to_rna = panels["adt"]["adt_to_rna"]
isotypes   = panels["adt"]["isotype_controls"]
adt_annot  = panels["adt"]["annotations"]
# targets = features that appear in the ADT signature table
targets = sorted(adt_sig["adt_feature"].unique())

print(f"RNA signature:  {sig.shape}")
print(f"ADT signature:  {adt_sig['perturbation'].nunique()} perts x "
      f"{adt_sig['adt_feature'].nunique()} features x "
      f"{adt_sig['condition'].nunique()} conditions "
      f"= {len(adt_sig):,} rows")

In [ ]:
# ---- rebuild quad: join RNA LFC with ADT Cohen's d -----------------------
quad_rows = []
for adt_feat, rna_genes in adt_to_rna.items():
    if adt_feat not in targets:
        continue
    rna_genes_in_sig = [g for g in rna_genes if g in sig.columns]
    if not rna_genes_in_sig:
        continue
    for cond in cond_order:
        adt_sub = adt_sig[(adt_sig["adt_feature"] == adt_feat) &
                          (adt_sig["condition"] == cond)]
        for _, arow in adt_sub.iterrows():
            pert = arow["perturbation"]
            if (pert, cond) not in sig.index:
                continue
            rna_lfc = sig.loc[(pert, cond), rna_genes_in_sig].mean()
            quad_rows.append({
                "perturbation": pert,
                "condition": cond,
                "adt_feature": adt_feat,
                "rna_lfc": float(rna_lfc),
                "adt_d": float(arow["cohens_d"]),
            })

quad = pd.DataFrame(quad_rows).dropna()
print(f"quadrant points: {len(quad):,}")

In [ ]:
# =============================== FIGURE ====================================
# A) Self-knockdown heatmap: for the 12 targets whose gene is in both
#    modalities, Cohen's d of the perturbation on its own ADT feature.
#    Validates the ADT signature matrix before using it more broadly.
#
# B) Quadrant plot: z-scored RNA LFC vs z-scored ADT Cohen's d for all
#    (perturbation x feature x condition) combinations.
#    Z-scoring puts both axes on the same scale so the 1:1 diagonal is
#    meaningful: points on it mean protein tracked RNA equally. Points
#    below = protein buffered against transcript. Points above = protein
#    amplified relative to transcript.

# ---- z-score both axes ----------------------------------------------------
quad["rna_lfc_z"] = (quad["rna_lfc"] - quad["rna_lfc"].mean()) / quad["rna_lfc"].std()
quad["adt_d_z"]   = (quad["adt_d"]   - quad["adt_d"].mean())   / quad["adt_d"].std()

print("z-score ranges:")
print(f"  RNA LFC z: {quad['rna_lfc_z'].min():.2f} to {quad['rna_lfc_z'].max():.2f}")
print(f"  ADT d z:   {quad['adt_d_z'].min():.2f} to {quad['adt_d_z'].max():.2f}")

# ---- build self-knockdown matrix for heatmap ------------------------------
self_kd_rows = []
for adt_feat, rna_genes in adt_to_rna.items():
    if adt_feat not in targets:
        continue
    for gene in rna_genes:
        sub = adt_sig[(adt_sig["perturbation"] == gene) &
                      (adt_sig["adt_feature"] == adt_feat)]
        if sub.empty:
            continue
        for cond in cond_order:
            row = sub[sub["condition"] == cond]
            self_kd_rows.append({
                "label": f"{gene} → {adt_feat}",
                "condition": cond,
                "cohens_d": row["cohens_d"].iloc[0] if not row.empty else np.nan,
            })

self_kd_mat = (pd.DataFrame(self_kd_rows)
               .pivot(index="label", columns="condition", values="cohens_d")
               .reindex(columns=cond_order))
self_kd_mat = self_kd_mat.reindex(
    self_kd_mat.mean(axis=1).sort_values().index)

# =============================== FIGURE ====================================
fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(18, 7.5),
    gridspec_kw={"width_ratios": [1, 1.6]},
)

# ---------------------------------- A --------------------------------------
vmax = np.nanmax(np.abs(self_kd_mat.values))
sns.heatmap(self_kd_mat, ax=ax["A"], cmap="RdBu_r", center=0,
            vmin=-vmax, vmax=vmax, annot=True, fmt="+.2f",
            annot_kws={"size": 8}, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Cohen's d", "shrink": 0.7})
ax["A"].tick_params(axis="y", labelsize=8, rotation=0)
ax["A"].tick_params(axis="x", labelsize=9, rotation=25)

# ---------------------------------- B --------------------------------------
lim = max(abs(quad["rna_lfc_z"].max()), abs(quad["adt_d_z"].max())) * 1.08

# 1:1 diagonal — meaningful after z-scoring
ax["B"].plot([-lim, lim], [-lim, lim], ls="--", c="grey", lw=1,
             zorder=1, label="1:1  protein tracks RNA")
ax["B"].axhline(0, c="k", lw=0.5, ls=":", zorder=1)
ax["B"].axvline(0, c="k", lw=0.5, ls=":", zorder=1)

# background cloud — all points
for cond in cond_order:
    d = quad[quad["condition"] == cond]
    alpha = np.clip(np.abs(d["adt_d"]).values / 1.5, 0.12, 0.75)
    ax["B"].scatter(d["rna_lfc_z"], d["adt_d_z"], s=10, alpha=alpha,
                    color=pal.get(cond, "#888"), linewidths=0,
                    rasterized=True, label=cond)

# self-knockdown points on top with black outline
texts = []
for adt_feat, rna_genes in adt_to_rna.items():
    for gene in rna_genes:
        sub = quad[(quad["perturbation"] == gene) &
                   (quad["adt_feature"] == adt_feat)]
        if sub.empty:
            continue
        for _, r in sub.iterrows():
            ax["B"].scatter(r["rna_lfc_z"], r["adt_d_z"], s=55, zorder=5,
                            color=pal.get(r["condition"], "#888"),
                            edgecolor="black", linewidth=0.9)
            texts.append(ax["B"].text(
                r["rna_lfc_z"], r["adt_d_z"],
                f"{gene}|{r['condition'][:3]}",
                fontsize=6.5, zorder=6))

adjust_text(texts, ax=ax["B"],
            arrowprops=dict(arrowstyle="-", lw=0.4, color="grey"),
            expand=(1.3, 1.5), force_text=(0.5, 0.7),
            ensure_inside_axes=False)

ax["B"].set_xlim(-lim, lim)
ax["B"].set_ylim(-lim, lim)
ax["B"].set_xlabel("RNA log2FC  (z-scored across all contrasts)", fontsize=10)
ax["B"].set_ylabel("ADT Cohen's d  (z-scored across all contrasts)", fontsize=10)
ax["B"].legend(fontsize=8, loc="center right", framealpha=0.9,
               markerscale=3)

titles = {
    "A": "ADT self-knockdown (Cohen's d)",
    "B": "z-scored RNA LFC vs ADT Cohen's d — all perturbations × features",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles[label]}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "07_adt_quadrant", cfg)

In [ ]:
print("sig index type:", type(sig.index))
print("adt_sig columns:", adt_sig.columns.tolist())
print("targets:", len(targets))
print("adt_to_rna keys:", len(adt_to_rna))